<div style="display: flex; align-items: center;">
    <h1>Differentiable irrigation control with diffWOFOST</h1>
    <img src="https://raw.githubusercontent.com/WUR-AI/diffWOFOST/refs/heads/main/docs/logo/diffwofost.png" width="150" style="margin-left: 20px;">
</div>

A final opportunity for differentiable crop models is not only to improve the model itself, but to optimize management decisions inside it. Irrigation is already part of the system dynamics of WOFOST. Once the model is differentiable, an economic objective can send gradients back through the water balance and into the rule that decided *when* to irrigate and *how much* to apply.

This notebook keeps water-limited WOFOST unchanged and injects a daily irrigation amount $a_t$ (mm) into the freely draining water balance. Three controllers are compared on the same euro-per-hectare reward:

- a **closed-loop policy** $\pi_\omega(s_t, x_t)$ that reads soil moisture, crop stage and rainfall, and fires discrete-looking irrigation *events*;
- an **open-loop vector** $(u_0,\ldots,u_{T-1})$ that ignores state, starts from near zero, and is specific to this weather year;
- a **no-stress rule** that irrigates only to keep WOFOST's transpiration reduction `RFTRA` at 1.

$$
s_{t+1} = f_{\mathrm{PBM}}\bigl(s_t, x_t, a_t, \theta\bigr), \qquad a_t = \pi_\omega(s_t, x_t) \;\text{or}\; a_t = u_t.
$$

Here $s_t$ is the crop and soil state, $x_t$ is the weather, $\theta$ are the usual WOFOST parameters, and $\omega$ or $u$ are the learnable controls. The reward is beet revenue minus irrigation cost. Irrigation is not treated as inherently bad: the no-stress rule shows what it costs to close the yield gap completely.


## 1. Why differentiable control is useful

This control perspective has three practical advantages.

1. **Transparency.** The consequences of an irrigation decision can be traced through soil moisture, transpiration reduction (`RFTRA`), assimilation, and storage-organ growth. That is a more mechanistic account than optimizing a policy against an opaque reward.

2. **Data efficiency.** Gradient-based updates provide a directional signal for changing the policy. The optimizer does not have to discover useful irrigations by randomly exploring state–action trajectories.

3. **Coupling to physiology.** The policy remains inside the water balance, so it is automatically constrained by infiltration capacity, percolation, crop presence, and the rest of the WOFOST dynamics.

There are also real difficulties, which this notebook will run into in miniature. Early-season irrigations affect yield months later, so the gradient signal can be weak. European sugar-beet irrigation is *event-based* — the soil dries for several days, then 15–30 mm is applied in one go — whereas a naive differentiable controller emits a continuous daily trickle. The closed-loop policy therefore uses a dryness threshold and a straight-through estimator so that the forward pass looks like a farmer event while Adam still receives a gradient. The open-loop vector has no soil-moisture structure: each day is an off/on event with a learnable dose, initialized off, so Adam has to discover the calendar rather than refine a closed-loop warm start.


## 2. Software requirements

Install the latest `diffwofost` if needed. The notebook also uses `pandas` and `matplotlib`. The water-limited sugar-beet test case is downloaded from the PCSE repository, following the same pattern as the other example notebooks.


In [ ]:
%%capture
# install required packages when needed
!pip install -q diffwofost matplotlib pandas


In [1]:
%matplotlib inline

from pathlib import Path
import urllib.request
import warnings

import matplotlib
matplotlib.style.use("ggplot")
import matplotlib.pyplot as plt
import pandas as pd
import torch

from diffwofost.physical_models.config import ComputeConfig, Configuration
from diffwofost.physical_models.crop.wofost72 import Wofost72
from diffwofost.physical_models.engine import Engine
from diffwofost.physical_models.soil.classic_waterbalance import WaterbalanceFD, WaterbalancePP
from diffwofost.physical_models.test import get_test_data, prepare_engine_input

warnings.filterwarnings("ignore", message="To copy construct from a tensor.*")
ComputeConfig.set_device("cpu")
ComputeConfig.set_dtype(torch.float64)

print(f"torch version: {torch.__version__}")
print(f"device: {ComputeConfig.get_device()}")
print(f"dtype: {ComputeConfig.get_dtype()}")


torch version: 2.11.0+cu130
device: cpu
dtype: torch.float64


## 3. A water-limited sugar-beet season

We use PCSE's water-limited WOFOST 7.2 regression case `test_waterlimitedproduction_wofost72_03.yaml`: sugar beet sown on 27 March 2010. The freely draining water balance (`WaterbalanceFD`) lets soil moisture evolve from rainfall, transpiration and evaporation, so drought can reduce `RFTRA` and therefore assimilation.

Two reference runs bound the problem:

- **Rainfed water-limited production**: no irrigation.
- **Potential production**: the same crop with soil moisture held at field capacity. This is the physiological yield ceiling if water were never limiting, and it is *not* an irrigated operating plan (the irrigation bill is zero by construction).

WOFOST reports `TWSO` as kg/ha of *dry* storage-organ biomass. Fresh beet yield is obtained with a dry-matter fraction of 0.23, a typical value for sugar beet:

$$
Y_{\mathrm{fresh}} = \frac{\mathrm{TWSO}}{1000 \times 0.23}\quad[\mathrm{t/ha}].
$$

That conversion matters for the economics below. This test cultivar and site reach about 50 t/ha fresh under potential production, which is below a 90 t/ha commercial crop; euro figures in this notebook are therefore for *this* WOFOST case, not for a high-yielding European contract.


In [2]:
filename = "test_waterlimitedproduction_wofost72_03.yaml"
url = (
    "https://raw.githubusercontent.com/ajwdewit/pcse/refs/heads/master/"
    f"tests/test_data/{filename}"
)
test_data_path = Path(filename)
if not test_data_path.exists():
    urllib.request.urlretrieve(url, test_data_path)
    print(f"Downloaded: {test_data_path.name}")
else:
    print(f"Using local file: {test_data_path.name}")

test_data = get_test_data(test_data_path)

crop_model_params = [
    "SPAN", "TDWI", "TBASE", "PERDL", "RGRLAI", "KDIFTB", "SLATB",
    "TSUMEM", "TBASEM", "TEFFMX", "TSUM1", "TSUM2", "DLO", "DLC", "DVSI", "DVSEND", "DTSMTB",
    "AMAXTB", "EFFTB", "TMPFTB", "TMNFTB",
    "Q10", "RMR", "RML", "RMS", "RMO", "RFSETB",
    "CFET", "DEPNR", "IAIRDU", "IOX", "CRAIRC", "SM0", "SMW", "SMFCF", "WAV",
    "RDI", "RRI", "RDMCR", "RDMSOL", "RDRRTB",
    "RDRSTB", "SSATB", "SPA",
    "FRTB", "FLTB", "FSTB", "FOTB",
    "CVL", "CVO", "CVR", "CVS",
]
provider, weather, agromanagement, _ = prepare_engine_input(test_data, crop_model_params)

print(f"crop: {test_data['ModelParameters'].get('CRPNAM', 'unknown')}")
print(f"WAV (initial available water): {float(provider['WAV']):.1f} cm")
print(
    "soil moisture bounds SMW / SMFCF / SM0: "
    f"{float(provider['SMW']):.3f} / {float(provider['SMFCF']):.3f} / {float(provider['SM0']):.3f}"
)


Using local file: test_waterlimitedproduction_wofost72_03.yaml
crop: Sugar beets
WAV (initial available water): 10.0 cm
soil moisture bounds SMW / SMFCF / SM0: 0.151 / 0.318 / 0.415


In [3]:
wlp_config = Configuration(
    CROP=Wofost72,
    SOIL=WaterbalanceFD,
    OUTPUT_VARS=["DVS", "LAI", "SM", "TAGP", "TRA", "RFTRA", "TWSO", "TWST", "RD"],
)

pp_config = Configuration(
    CROP=Wofost72,
    SOIL=WaterbalancePP,
    OUTPUT_VARS=["DVS", "LAI", "SM", "TAGP", "RFTRA", "TWSO"],
)


def scalarize(value):
    if isinstance(value, torch.Tensor):
        return float(value.detach().cpu())
    return value


def results_to_frame(results, irrigation=None):
    rows = []
    for row in results:
        rows.append({key: (value if key == "day" else scalarize(value)) for key, value in row.items()})
    frame = pd.DataFrame(rows)
    frame["day"] = pd.to_datetime(frame["day"])
    frame = frame.set_index("day")
    if irrigation is not None:
        irr = pd.Series(
            {day: 10.0 * scalarize(amount) for day, amount in irrigation},
            name="irrigation",
        )
        irr.index = pd.to_datetime(irr.index)
        frame = frame.join(irr, how="left")
        frame["irrigation"] = frame["irrigation"].fillna(0.0)
    return frame


def summarize(name, frame, totirr=0.0):
    emerged = frame[frame["DVS"] > 0]
    n_irr = int((frame.get("irrigation", pd.Series(0, index=frame.index)) > 15.0).sum())
    print(
        f"{name:22s}  TWSO={frame['TWSO'].iloc[-1]:8.1f} kg/ha  "
        f"TOTIRR={totirr:5.1f} cm  min RFTRA={emerged['RFTRA'].min():.2f}  "
        f"events>15 mm={n_irr:3d}"
    )


rainfed_engine = Engine(config=wlp_config)
rainfed_engine.setup(provider, weather, agromanagement)
rainfed_engine.run_till_terminate()
rainfed_df = results_to_frame(rainfed_engine.get_output())

pp_engine = Engine(config=pp_config)
pp_engine.setup(provider, weather, agromanagement)
pp_engine.run_till_terminate()
pp_df = results_to_frame(pp_engine.get_output())

summarize("rainfed", rainfed_df, totirr=0.0)
summarize("potential production", pp_df, totirr=float("nan"))
print(
    "yield gap due to water: "
    f"{pp_df['TWSO'].iloc[-1] - rainfed_df['TWSO'].iloc[-1]:.0f} kg/ha dry  "
    f"({(pp_df['TWSO'].iloc[-1] - rainfed_df['TWSO'].iloc[-1]) / 1000 / 0.23:.1f} t/ha fresh at 23% DM)"
)


rainfed                 TWSO=  1979.6 kg/ha  TOTIRR=  0.0 cm  min RFTRA=0.00  events>15 mm=  0
potential production    TWSO= 12180.3 kg/ha  TOTIRR=  nan cm  min RFTRA=1.00  events>15 mm=  0
yield gap due to water: 10201 kg/ha dry  (44.4 t/ha fresh at 23% DM)


The rainfed crop loses a large fraction of potential storage-organ biomass. Soil moisture falls well below field capacity during canopy expansion, `RFTRA` drops, and growth of the beet (`TWSO`) stalls. Closing that gap is valuable only when the extra beet revenue exceeds the cost of the irrigation events that produced it. A fixed calendar such as "irrigate every 7 days" is a poor abstraction here: rainfall, soil water-holding capacity and crop demand jointly determine whether the next event is needed at all.


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(9, 9), sharex=True)

axes[0].plot(rainfed_df.index, rainfed_df["SM"], label="rainfed")
axes[0].plot(pp_df.index, pp_df["SM"], label="potential", linestyle="--")
axes[0].axhline(float(provider["SMFCF"]), color="0.4", linewidth=0.8, linestyle=":")
axes[0].axhline(float(provider["SMW"]), color="0.4", linewidth=0.8, linestyle=":")
axes[0].set_ylabel("SM $(-)$")
axes[0].legend()

axes[1].plot(rainfed_df.index, rainfed_df["RFTRA"], label="rainfed")
axes[1].plot(pp_df.index, pp_df["RFTRA"], label="potential", linestyle="--")
axes[1].set_ylabel("RFTRA $(-)$")

axes[2].plot(rainfed_df.index, rainfed_df["TWSO"], label="rainfed")
axes[2].plot(pp_df.index, pp_df["TWSO"], label="potential", linestyle="--")
axes[2].set_ylabel("TWSO (kg ha$^{-1}$)")
axes[2].set_xlabel("day")

fig.suptitle("Water-limited rainfed crop versus potential production", y=0.99)
fig.tight_layout()
plt.show()


## 4. Irrigation inside the water balance

In `WaterbalanceFD`, an irrigation event sets the effective irrigation rate `_RIRR` (applied amount times application efficiency). That rate is added to infiltrating water, updates root-zone moisture `SM`, and therefore changes the transpiration reduction factor `RFTRA` used by assimilation.

PCSE normally fires irrigation through an agromanagement signal. Here we do the same thing from Python: after the daily weather and management step, and before `calc_rates`, the policy writes a tensor into `engine.soil._RIRR`. Because that tensor is produced by $\pi_\omega$, backpropagation can flow from final `TWSO` through the water balance and into the policy weights.

Application efficiency is set to 0.7, a typical value for sprinkler irrigation. The policy and the cost model are expressed in **applied millimetres** (what the farmer decides). `_RIRR` and `TOTIRR` use WOFOST's centimetre units; `TOTIRR` is the *effective* amount that entered the soil.


## 5. Event-based irrigation as an action

Realistic sugar-beet irrigation in Europe is not a continuous water supply. Soil water is depleted over several days; once available water falls below a threshold, a substantial amount is applied in one event, soil moisture jumps, and the crop waits for depletion again. Rainfall can postpone or cancel the next event. Irrigation scientists call this **management allowed depletion** (MAD): allow a fraction of plant-available water to be used, then refill.

```
soil water
100% |───────╮       ╭────────╮
             │       │        │
 70% |       ╰───────╯        ╰──────
             ↑                 ↑
          25 mm             25 mm
          irrigation        irrigation
```

Sprinkler events in European datasets are typically around 15–20 mm (about 18 mm in Niedersachsen, with large field-to-field variation), often 7–9 times per season in some regions. Sugar-beet experiments in Austria have used 25–40 mm triggered by soil-water measurements; a German modelling study used 20 mm once a soil-water threshold was crossed. For an RL or differentiable-control environment, a daily action in $\{0, 10, 20, 30\}$ mm, or the continuous interval $[0, 30]$ mm, is therefore a defensible range. Drip systems apply smaller, more frequent amounts; they are not the baseline for a general European sugar-beet field.

The closed-loop policy is that MAD rule, not a free neural net from scratch. A dryness threshold, a rain skip, and a DVS window are multiplied into one sigmoid and turned into an event with a straight-through estimator. The applied amount is `min(dose, root-zone deficit / efficiency)`, so a late-season 25 mm pulse is not forced into a nearly full profile.

The late-DVS stop is **learnable**, not a hardcoded senescence law. It starts *open* (through harvest), and the residual MLP's last layer starts at zero, so step 0 is only the agronomic MAD prior: irrigate ~25 mm when the soil is dry and the crop has emerged. Adam has to discover for itself whether a late cutoff pays. In WOFOST, DVS 1–2 is filling rather than "crop over"; whether late events are worth it is left to $R$.

The dryness threshold, event size, and DVS-end are stored in unconstrained logit space and mapped to $(0, 1)$, $(0, 35]$ mm and $[1.2, 2.4]$, so they stay in range during gradient descent.


In [4]:
IRRIGATION_EFFICIENCY = 0.7
MAX_DOSE_MM = 35.0
MAX_DOSE_CM = MAX_DOSE_MM / 10.0
DM_FRACTION = 0.23          # fresh beet (t/ha) = TWSO (kg/ha) / 1000 / DM
BEET_PRICE = 40.0           # € / t fresh; typical NW-EU contract (2024 ~€38–47)
C_MM = 1.0                  # € / mm / ha for water + pumping energy
C_EVENT = 25.0              # € / ha labour/tractor to apply one hose-reel set
EVENT_MM_MIN = 5.0          # applied mm that counts as an event
EVENT_TEMP_MM = 0.25

from diffwofost.physical_models.crop.evapotranspiration import SWEAF


def kiosk_get(engine, name, default):
    if name in engine.kiosk:
        value = engine.kiosk[name]
        if value is not None:
            return value
    return default


def as_scalar(value, like):
    tensor = torch.as_tensor(value, dtype=like.dtype, device=like.device)
    return tensor.reshape(()) if tensor.numel() == 1 else tensor


def straight_through_gate(soft):
    """Binary in the forward pass; sigmoid gradient in the backward pass."""
    hard = (soft > 0.5).to(dtype=soft.dtype)
    return hard.detach() + soft - soft.detach()


def stacked_amounts(irrigation):
    return torch.stack([amount for _, amount in irrigation])


def fresh_yield_t_ha(twso):
    return twso / 1000.0 / DM_FRACTION


def irrigation_economics(twso, amounts_cm):
    """R = P Y - C_event N - c_mm I, with a differentiable event count."""
    y_fresh = fresh_yield_t_ha(twso)
    revenue = BEET_PRICE * y_fresh
    applied_mm = amounts_cm * 10.0
    events = torch.sigmoid((applied_mm - EVENT_MM_MIN) / EVENT_TEMP_MM)
    n_events = events.sum()
    cost = C_EVENT * n_events + C_MM * applied_mm.sum()
    reward = revenue - cost
    n_hard = int((applied_mm.detach() > 15.0).sum())
    return {
        "y_fresh": y_fresh,
        "revenue": revenue,
        "cost": cost,
        "reward": reward,
        "applied_mm": applied_mm.sum(),
        "n_events": n_events,
        "n_events_hard": n_hard,
    }


def episode_reward(twso, amounts_cm=None, *, potential=False):
    """Seasonal €/ha. Potential production has no irrigation bill."""
    revenue = BEET_PRICE * scalarize(fresh_yield_t_ha(twso))
    if potential:
        return revenue
    if amounts_cm is None:
        return revenue
    return scalarize(irrigation_economics(twso, amounts_cm)["reward"])


class IrrigationPolicy(torch.nn.Module):
    """MAD-style events plus a zero-init residual net.

    Management allowed depletion (MAD): wait until relative available water
    falls below a threshold, then apply a pulse up to the root-zone deficit.
    The late-DVS stop is learnable and starts open (through harvest).
    A small MLP, started at zero, can raise or lower the trigger.
    Step 0 is an agronomic MAD rule, not a previously optimized policy.
    """

    dvs_end_min = 1.2
    dvs_end_max = 2.4

    def __init__(
        self,
        max_dose_mm=MAX_DOSE_MM,
        temperature=0.05,
        init_threshold=0.45,
        init_dose_mm=25.0,
        init_dvs_end=2.4,
        hidden=16,
    ):
        super().__init__()
        self.max_dose_cm = max_dose_mm / 10.0
        self.temperature = temperature
        dtype = ComputeConfig.get_dtype()
        device = ComputeConfig.get_device()
        self.threshold_raw = torch.nn.Parameter(
            torch.logit(torch.tensor(init_threshold, dtype=dtype, device=device).clamp(1e-4, 1 - 1e-4))
        )
        self.dose_raw = torch.nn.Parameter(
            torch.logit(
                torch.tensor(init_dose_mm / max_dose_mm, dtype=dtype, device=device).clamp(1e-4, 1 - 1e-4)
            )
        )
        dvs_end_frac = (init_dvs_end - self.dvs_end_min) / (self.dvs_end_max - self.dvs_end_min)
        self.dvs_end_raw = torch.nn.Parameter(
            torch.logit(torch.tensor(dvs_end_frac, dtype=dtype, device=device).clamp(1e-4, 1 - 1e-4))
        )
        self.residual = torch.nn.Sequential(
            torch.nn.Linear(5, hidden, dtype=dtype, device=device),
            torch.nn.Tanh(),
            torch.nn.Linear(hidden, 1, dtype=dtype, device=device),
        )
        torch.nn.init.zeros_(self.residual[-1].weight)
        torch.nn.init.zeros_(self.residual[-1].bias)

    def decoded(self):
        threshold = torch.sigmoid(self.threshold_raw)
        dose_cm = self.max_dose_cm * torch.sigmoid(self.dose_raw)
        dvs_end = self.dvs_end_min + (self.dvs_end_max - self.dvs_end_min) * torch.sigmoid(
            self.dvs_end_raw
        )
        return threshold, dose_cm, dvs_end

    def features(self, rel_sm, dvs, rain, lai, et0):
        return torch.stack(
            [
                rel_sm,
                dvs / 2.0,
                rain,
                lai / 5.0,
                et0 / 0.5,
            ]
        )

    def forward(self, engine):
        sm = engine.soil.states.SM
        smw = engine.soil.params.SMW
        smfc = engine.soil.params.SMFCF
        rel_sm = (sm - smw) / (smfc - smw).clamp_min(1e-6)
        rain = as_scalar(engine.drv.RAIN, sm)
        dvs = as_scalar(kiosk_get(engine, "DVS", torch.zeros_like(sm)), sm)
        lai = as_scalar(kiosk_get(engine, "LAI", torch.zeros_like(sm)), sm)
        et0 = as_scalar(engine.drv.ET0, sm)
        rd = as_scalar(
            kiosk_get(engine, "RD", torch.tensor(10.0, dtype=sm.dtype, device=sm.device)),
            sm,
        ).clamp_min(1.0)
        threshold, dose_cm, dvs_end = self.decoded()
        deficit_cm = torch.clamp((smfc - sm) * rd, min=0.0)
        applied_cap = deficit_cm / IRRIGATION_EFFICIENCY

        tree = torch.sigmoid((threshold - rel_sm) / self.temperature)
        tree = tree * torch.sigmoid((0.80 - rain) / 0.15)
        tree = tree * torch.sigmoid((dvs - 0.12) / 0.06)
        tree = tree * torch.sigmoid((dvs_end - dvs) / 0.08)
        delta = self.residual(self.features(rel_sm, dvs, rain, lai, et0)).reshape(())
        trigger = torch.sigmoid(torch.logit(tree.clamp(1e-4, 1.0 - 1e-4)) + 2.0 * delta)
        dose = torch.minimum(dose_cm, applied_cap)
        return straight_through_gate(trigger) * dose


class NoStressPolicy(torch.nn.Module):
    """Keep RFTRA = 1: refill toward field capacity before SM falls below SMCR.

    WOFOST sets RFTRA < 1 when root-zone moisture is below the critical content
    SMCR (a function of ET0 and DEPNR). This rule is not trained. It is a
    physiological baseline: prevent water-limited transpiration, ignore cost.
    Irrigation is applied one day early so that today's RFTRA, which is computed
    from morning SM, never sees the deficit.
    """

    def forward(self, engine):
        sm = engine.soil.states.SM
        smw = engine.soil.params.SMW
        smfc = engine.soil.params.SMFCF
        params = engine.crop.evtra.etmodule.params
        rd = as_scalar(
            kiosk_get(engine, "RD", torch.tensor(10.0, dtype=sm.dtype, device=sm.device)),
            sm,
        ).clamp_min(1.0)
        dvs = as_scalar(kiosk_get(engine, "DVS", torch.zeros_like(sm)), sm)
        et0 = as_scalar(engine.drv.ET0, sm)
        et0_crop = torch.clamp(params.CFET * et0, min=0.0)
        swdep = SWEAF(et0_crop, params.DEPNR)
        smcr = (1.0 - swdep) * (smfc - smw) + smw
        sm_end = sm - 2.5 * et0_crop / rd
        deficit_cm = torch.clamp((smfc - sm) * rd, min=0.0)
        applied_cm = torch.clamp(deficit_cm / IRRIGATION_EFFICIENCY, max=5.0)
        crop = (dvs > 0).to(dtype=sm.dtype)
        needed = (sm_end < smcr).to(dtype=sm.dtype)
        return needed * crop * applied_cm




def run_with_policy(engine, policy=None, efficiency=IRRIGATION_EFFICIENCY):
    """Advance a prepared engine, injecting irrigation (cm) before each rate calculation."""
    zero = torch.zeros((), dtype=ComputeConfig.get_dtype(), device=ComputeConfig.get_device())
    irrigation = []
    while engine.flag_terminate is False:
        engine.day, delt = engine.timer()
        engine.integrate(engine.day, delt)
        engine.drv = engine._get_driving_variables(engine.day)
        engine.agromanager(engine.day, engine.drv)
        amount_cm = zero if policy is None else policy(engine)
        engine.soil._RIRR = amount_cm * efficiency
        irrigation.append((engine.day, amount_cm))
        engine.calc_rates(engine.day, engine.drv)
        if engine.flag_terminate is True:
            engine._terminate_simulation(engine.day)
    return irrigation


def evaluate_policy(policy, provider, weather, agromanagement, config):
    if policy is not None and hasattr(policy, "reset"):
        policy.reset()
    engine = Engine(config=config)
    engine.setup(provider, weather, agromanagement)
    irrigation = run_with_policy(engine, policy=policy)
    results = engine.get_output()
    amounts_cm = stacked_amounts(irrigation)
    twso = results[-1]["TWSO"]
    totirr = engine.soil.states.TOTIRR
    eco = irrigation_economics(twso, amounts_cm)
    return {
        "engine": engine,
        "results": results,
        "frame": results_to_frame(results, irrigation),
        "irrigation": irrigation,
        "amounts_cm": amounts_cm,
        "twso": twso,
        "totirr": totirr,
        **eco,
    }


The daily loop is the ordinary PCSE step, with one extra line that writes $a_t = \pi_\omega(s_t, x_t)$ into the water balance. That is the implementation of the decision-making box: the policy is inside $f_{\mathrm{PBM}}$, not wrapped around it as an external simulator. `_RIRR` is in cm; plots and the cost model convert to mm (1 mm = 10 m³/ha).


In [5]:
def summarize_economics(name, outcome=None, frame=None, amounts_cm=None, potential=False):
    if frame is None:
        frame = outcome["frame"]
    twso = frame["TWSO"].iloc[-1] if outcome is None else outcome["twso"]
    if potential:
        y = scalarize(fresh_yield_t_ha(twso))
        print(
            f"{name:28s}  fresh={y:5.1f} t/ha  applied=   0 mm  events=  0  "
            f"revenue={BEET_PRICE * y:7.0f} €/ha  cost=    0  reward={BEET_PRICE * y:7.0f} €/ha"
        )
        return
    if outcome is not None:
        eco = outcome
        amounts = outcome["amounts_cm"]
    else:
        amounts = amounts_cm
        eco = irrigation_economics(torch.as_tensor(twso, dtype=ComputeConfig.get_dtype()), amounts)
    y = scalarize(eco["y_fresh"])
    mm = scalarize(eco["applied_mm"])
    n_hard = eco["n_events_hard"] if outcome is not None else int((amounts.detach() * 10 > 15).sum())
    print(
        f"{name:28s}  fresh={y:5.1f} t/ha  applied={mm:5.0f} mm  events={n_hard:3d}  "
        f"revenue={scalarize(eco['revenue']):7.0f} €/ha  "
        f"cost={scalarize(eco['cost']):5.0f}  reward={scalarize(eco['reward']):7.0f} €/ha"
    )


zero_amounts = torch.zeros(len(rainfed_df), dtype=ComputeConfig.get_dtype())
initial_policy = IrrigationPolicy()
initial_threshold, initial_dose, initial_dvs_end = initial_policy.decoded()
print(
    "initial MAD policy: up to "
    f"{10 * initial_dose.item():.0f} mm when relative soil moisture < {initial_threshold.item():.2f}, "
    f"crop emerged until DVS {initial_dvs_end.item():.2f}, residual net at 0"
)

initial = evaluate_policy(initial_policy, provider, weather, agromanagement, wlp_config)
summarize_economics("initial policy", initial)
summarize_economics("rainfed", frame=rainfed_df, amounts_cm=zero_amounts)
summarize_economics("potential production", frame=pp_df, potential=True)

no_stress = evaluate_policy(NoStressPolicy(), provider, weather, agromanagement, wlp_config)
emerged = no_stress["frame"]["DVS"] > 0
print(
    "no-stress min RFTRA="
    f"{no_stress['frame'].loc[emerged, 'RFTRA'].min():.3f}  "
    f"(potential production is {pp_df.loc[pp_df['DVS'] > 0, 'RFTRA'].min():.3f} by construction)"
)
summarize_economics("no-stress (RFTRA=1)", no_stress)


initial MAD policy: up to 25 mm when relative soil moisture < 0.45, crop emerged until DVS 2.40, residual net at 0
initial policy                fresh= 27.4 t/ha  applied=  400 mm  events= 16  revenue=   1095 €/ha  cost=  800  reward=    295 €/ha
rainfed                       fresh=  8.6 t/ha  applied=    0 mm  events=  0  revenue=    344 €/ha  cost=    0  reward=    344 €/ha
potential production          fresh= 53.0 t/ha  applied=   0 mm  events=  0  revenue=   2118 €/ha  cost=    0  reward=   2118 €/ha
no-stress min RFTRA=1.000  (potential production is 1.000 by construction)
no-stress (RFTRA=1)           fresh= 53.0 t/ha  applied=  785 mm  events= 30  revenue=   2119 €/ha  cost= 1906  reward=    213 €/ha


## 6. An economic reward, not a water penalty

Irrigation is not inherently bad. The agent should irrigate when the extra beet value is expected to exceed the cost of the event.

The earlier price of €2 per mm plus €20 per event was too harsh: it billed labour twice. A hose-reel with a rain-gun applying 25 mm in England costs about £53/ha all-in (labour, tractor, water and diesel; capital assumed sunk). That is roughly €50–55/ha per application, or €2.0–2.2 per mm if the whole bill is treated as a volume price.

Splitting the bill into a mobilisation fee and a cheaper volume term is closer to how the cost actually arises:

$$
C(I, N) = C_{\mathrm{event}}\,N + c_{\mathrm{mm}}\,I,
\qquad
R = P\,Y_{\mathrm{fresh}} - C(I, N).
$$

| assumption | value | reason |
|---|---|---|
| beet price `P` | €40/t | typical NW-EU contract; 2024 was about €38–47, high years ~€50 |
| volume cost `c_mm` | €1.0 per mm per ha | water + pumping (1 mm = 10 m³/ha) |
| event cost `C_event` | €25/ha | labour/tractor to apply one set |
| dry-matter fraction | 0.23 | WOFOST `TWSO` is dry biomass |

A 25 mm event then costs €25 + €25 = €50/ha, in line with the BBRO figure, without double-counting. One hundred millimetres in four events costs €200 + €100 = €300/ha, not €400–500. Capital (borehole, pump, pipes, reel) is excluded, as in that study: it is a sunk cost if the kit is already used on potatoes or vegetables.

Adam minimizes $J = -R$. Comparison tables report $R$ in €/ha. Potential production is listed as beet *revenue* with a zero irrigation bill. The no-stress rule is the other bound: it pays the irrigation bill that actually keeps `RFTRA` at 1.

Because $Y$, $I$ and $N$ are all produced on the same gradient tape as the water balance, a standard Adam step on $\omega$ is enough.


In [6]:
torch.manual_seed(0)
policy = IrrigationPolicy()
# All parameters from step 0. DVS-end starts open (harvest), residual at zero:
# this is MAD, not a warm start from a previously optimized €830 policy.
# Residual uses a smaller lr because a 16-unit net at 0.08 instantly turns
# the STE gate off and the tape collapses to rainfed.
tree_params = [policy.threshold_raw, policy.dose_raw, policy.dvs_end_raw]
residual_params = list(policy.residual.parameters())
optimizer = torch.optim.Adam(
    [
        {"params": tree_params, "lr": 0.08},
        {"params": residual_params, "lr": 0.005},
    ]
)
history = []
best = None
best_loss = float("inf")
n_steps = 80

for step in range(n_steps):
    optimizer.zero_grad()
    outcome = evaluate_policy(policy, provider, weather, agromanagement, wlp_config)
    loss = -outcome["reward"]
    threshold, dose_cm, dvs_end = policy.decoded()
    record = {
        "step": step,
        "loss": scalarize(loss),
        "reward": scalarize(outcome["reward"]),
        "y_fresh": scalarize(outcome["y_fresh"]),
        "applied_mm": scalarize(outcome["applied_mm"]),
        "n_events": outcome["n_events_hard"],
        "threshold": scalarize(threshold),
        "dose_mm": 10.0 * scalarize(dose_cm),
        "dvs_end": scalarize(dvs_end),
    }
    history.append(record)
    if record["loss"] < best_loss:
        best_loss = record["loss"]
        best = {key: value.detach().clone() for key, value in policy.state_dict().items()}

    loss.backward()
    optimizer.step()

    print(
        f"step {step:03d}  fresh={record['y_fresh']:5.1f} t/ha  "
        f"applied={record['applied_mm']:5.0f} mm  events={record['n_events']:3d}  "
        f"threshold={record['threshold']:.3f}  dose={record['dose_mm']:.1f} mm  "
        f"dvs_end={record['dvs_end']:.2f}  "
        f"R={record['reward']:7.0f} €/ha"
    )

policy.load_state_dict(best)
optimized = evaluate_policy(policy, provider, weather, agromanagement, wlp_config)
opt_threshold, opt_dose, opt_dvs_end = policy.decoded()
print(
    "\noptimized policy: apply "
    f"{10 * opt_dose.item():.0f} mm when relative soil moisture < {opt_threshold.item():.2f}, "
    f"until DVS {opt_dvs_end.item():.2f}"
)
summarize_economics("optimized policy", optimized)


step 000  fresh= 27.4 t/ha  applied=  400 mm  events= 16  threshold=0.450  dose=25.0 mm  dvs_end=2.40  R=    295 €/ha
step 001  fresh= 30.6 t/ha  applied=  435 mm  events= 17  threshold=0.470  dose=25.6 mm  dvs_end=2.40  R=    363 €/ha
step 002  fresh= 32.3 t/ha  applied=  444 mm  events= 17  threshold=0.490  dose=26.1 mm  dvs_end=2.40  R=    423 €/ha
step 003  fresh= 34.7 t/ha  applied=  479 mm  events= 18  threshold=0.509  dose=26.6 mm  dvs_end=2.40  R=    459 €/ha
step 004  fresh= 36.6 t/ha  applied=  488 mm  events= 18  threshold=0.529  dose=27.1 mm  dvs_end=2.40  R=    525 €/ha
step 005  fresh= 39.1 t/ha  applied=  524 mm  events= 19  threshold=0.548  dose=27.6 mm  dvs_end=2.40  R=    564 €/ha
step 006  fresh= 40.4 t/ha  applied=  561 mm  events= 20  threshold=0.567  dose=28.0 mm  dvs_end=2.40  R=    553 €/ha
step 007  fresh= 43.9 t/ha  applied=  598 mm  events= 21  threshold=0.587  dose=28.5 mm  dvs_end=2.40  R=    633 €/ha
step 008  fresh= 45.5 t/ha  applied=  607 mm  events= 21

In [ ]:
hist = pd.DataFrame(history)
rainfed_fresh = scalarize(fresh_yield_t_ha(rainfed_df["TWSO"].iloc[-1]))
pp_fresh = scalarize(fresh_yield_t_ha(pp_df["TWSO"].iloc[-1]))

fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
axes[0].plot(hist["step"], hist["y_fresh"], marker="o")
axes[0].axhline(pp_fresh, color="0.3", linestyle="--", label="potential")
axes[0].axhline(rainfed_fresh, color="0.3", linestyle=":", label="rainfed")
axes[0].set_xlabel("step")
axes[0].set_ylabel("fresh yield (t ha$^{-1}$)")
axes[0].legend(fontsize=8)

axes[1].plot(hist["step"], hist["reward"], marker="o", color="C2")
axes[1].axhline(episode_reward(rainfed_df["TWSO"].iloc[-1], zero_amounts), color="0.3", linestyle=":", label="rainfed")
axes[1].set_xlabel("step")
axes[1].set_ylabel("reward (€ ha$^{-1}$)")
axes[1].legend(fontsize=8)

axes[2].plot(hist["step"], hist["threshold"], marker="o", label="threshold")
axes[2].plot(hist["step"], hist["dose_mm"] / MAX_DOSE_MM, marker="s", label="dose / 35 mm")
axes[2].plot(hist["step"], hist["dvs_end"] / 2.4, marker="^", label="DVS-end / 2.4")
axes[2].set_xlabel("step")
axes[2].set_ylabel("decoded parameters")
axes[2].legend(fontsize=8)

fig.suptitle("Gradient updates on the irrigation policy", y=1.02)
fig.tight_layout()
plt.show()


## 7. What the optimized policy does

The learned rule can still be read: a relative-moisture threshold, an event size, a late-DVS stop, and a small residual correction. Because each event is (approximately) all-or-nothing and capped by the root-zone deficit, soil moisture should show the sawtooth of a farmer MAD schedule rather than a daily drizzle. Training starts as MAD through harvest, not at a previously optimized cutoff. If late millimetres do not pay, Adam has to lower DVS-end or use the residual to skip them.

The no-stress rule is the physiological reference: it refills toward field capacity whenever today's ET would take moisture below WOFOST's critical content `SMCR`, so `RFTRA` stays at 1 and yield approaches potential production. That uses a lot of water. At €40/t and €1/mm plus €25 per event, closing every last millimetre of drought stress is not automatically the economic optimum. Adam may leave residual stress if the last events do not pay. The checkpoint with the best $R$ is kept, because the straight-through gradient is a biased surrogate: later steps can walk off a discrete schedule into rainfed even when that true $R$ is worse.


In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(9, 11), sharex=True)

series = [
    ("rainfed", rainfed_df, "-"),
    ("initial policy", initial["frame"], "--"),
    ("optimized policy", optimized["frame"], "-"),
    ("no-stress (RFTRA=1)", no_stress["frame"], "-."),
    ("potential", pp_df, ":"),
]

for name, frame, ls in series:
    axes[0].plot(frame.index, frame["SM"], label=name, linestyle=ls)
    axes[1].plot(frame.index, frame["RFTRA"], label=name, linestyle=ls)
    axes[2].plot(frame.index, frame["LAI"], label=name, linestyle=ls)
    axes[3].plot(frame.index, frame["TWSO"], label=name, linestyle=ls)

opt_frame = optimized["frame"]
axes[0].bar(
    opt_frame.index,
    opt_frame["irrigation"] * 0.004,
    color="C0",
    alpha=0.35,
    width=1.0,
    label="irrigation (scaled mm)",
)
axes[0].axhline(float(provider["SMFCF"]), color="0.5", linewidth=0.7, linestyle=":")
axes[0].set_ylabel("SM $(-)$")
axes[0].legend(loc="upper right", fontsize=8)
axes[1].set_ylabel("RFTRA $(-)$")
axes[2].set_ylabel("LAI $(-)$")
axes[3].set_ylabel("TWSO (kg ha$^{-1}$)")
axes[3].set_xlabel("day")

fig.suptitle("Physiological pathway from irrigation to yield", y=0.99)
fig.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.2))
ax.bar(opt_frame.index, opt_frame["irrigation"], width=1.0, color="C0", alpha=0.85, label="optimized")
ax.plot(initial["frame"].index, initial["frame"]["irrigation"], color="C1", linewidth=0.8, alpha=0.7, label="initial")
ax.set_ylabel("irrigation (mm day$^{-1}$)")
ax.set_xlabel("day")
ax.legend()
ax.set_title("Daily irrigation events applied by the policy")
fig.tight_layout()
plt.show()


def comparison_row(name, twso, amounts_cm, potential=False):
    y = scalarize(fresh_yield_t_ha(twso))
    if potential:
        return {
            "fresh yield (t/ha)": y,
            "applied irrigation (mm)": 0.0,
            "events (>15 mm)": 0,
            "min RFTRA": 1.0,
            "revenue (€/ha)": BEET_PRICE * y,
            "irrigation cost (€/ha)": 0.0,
            "reward (€/ha)": BEET_PRICE * y,
        }
    eco = irrigation_economics(torch.as_tensor(scalarize(twso), dtype=ComputeConfig.get_dtype()), amounts_cm)
    return {
        "fresh yield (t/ha)": y,
        "applied irrigation (mm)": scalarize(eco["applied_mm"]),
        "events (>15 mm)": eco["n_events_hard"],
        "min RFTRA": float("nan"),
        "revenue (€/ha)": scalarize(eco["revenue"]),
        "irrigation cost (€/ha)": scalarize(eco["cost"]),
        "reward (€/ha)": scalarize(eco["reward"]),
    }


comparison = pd.DataFrame(
    [
        comparison_row("rainfed", rainfed_df["TWSO"].iloc[-1], zero_amounts),
        comparison_row("initial policy", initial["twso"], initial["amounts_cm"]),
        comparison_row("optimized policy", optimized["twso"], optimized["amounts_cm"]),
        comparison_row("no-stress (RFTRA=1)", no_stress["twso"], no_stress["amounts_cm"]),
        comparison_row("potential production", pp_df["TWSO"].iloc[-1], None, potential=True),
    ],
    index=["rainfed", "initial policy", "optimized policy", "no-stress (RFTRA=1)", "potential production"],
)
def with_min_rftra(frame, row_name, table):
    emerged = frame["DVS"] > 0
    table.loc[row_name, "min RFTRA"] = frame.loc[emerged, "RFTRA"].min()
    return table


comparison = with_min_rftra(rainfed_df, "rainfed", comparison)
comparison = with_min_rftra(initial["frame"], "initial policy", comparison)
comparison = with_min_rftra(optimized["frame"], "optimized policy", comparison)
comparison = with_min_rftra(no_stress["frame"], "no-stress (RFTRA=1)", comparison)
comparison.round(2)


The rainfed crop is the lower bound on yield and a serious economic baseline: it has no irrigation bill. The no-stress rule is the physiological upper bound among *feasible* irrigated plans: `RFTRA` stays at 1 and yield matches potential production, but the water and event bill is real. Potential production itself still has a zero irrigation bill by construction.

The optimized policy should sit between rainfed and no-stress on yield, and beat both on $R$ only if extra tonnes are worth more than `C_event N + c_mm I`. Raising `c_mm` or `C_event` moves the same architecture toward fewer, later events.

The policy is closed-loop: the same few weights can react to whatever soil moisture the season produces. The next section asks a narrower question: if this weather year is known in advance, how far can we get by optimizing the daily irrigation vector from scratch?


## 8. Open-loop schedule: optimize the daily irrigation vector

The policy above conditions on crop and soil state. A more brute-force alternative is to ignore that context and treat irrigation as a free vector for this weather year.

A *continuous* millimetre amount per day does not work with an event fee. Adam can raise a day from 0.2 mm to 4.6 mm, but the €25 mobilisation cost turns on sharply at 5 mm, so the optimizer stops short of an event and never irrigates. That is a barrier in the cost surface, not a lack of yield gradient.

Each day therefore has two parameters, with a straight-through gate so the forward pass is an event or nothing:

$$
a_t = g_t \cdot d_t, \qquad g_t \in \{0,1\}, \qquad d_t \in (0, 35]\,\mathrm{mm}.
$$

Gates start **off** (no closed-loop warm start, no daily drizzle). When a gate crosses 0.5, that day jumps to a ~25 mm pulse instead of walking through the 5 mm cliff. Dose is learnable per day. The same objective $R$ is maximized with Adam for several hundred steps — more than the closed-loop policy, because there are hundreds of weakly coupled gates. In principle this vector can overfit 2010 rainfall and beat any state-feedback rule on this episode.

The resulting schedule is an *episode-specific* open-loop plan: it is allowed to overfit this rainfall series, and it will not transfer to another year.


In [7]:
class OpenLoopSchedule(torch.nn.Module):
    """Daily open-loop events: each day is off or a full dose (straight-through)."""

    def __init__(self, n_days, max_dose_mm=MAX_DOSE_MM, init_dose_mm=25.0, init_gate_logit=-2.0):
        super().__init__()
        self.max_dose_cm = max_dose_mm / 10.0
        dtype = ComputeConfig.get_dtype()
        device = ComputeConfig.get_device()
        torch.manual_seed(0)
        self.gate_raw = torch.nn.Parameter(
            torch.full((n_days,), init_gate_logit, dtype=dtype, device=device)
            + 0.2 * torch.randn(n_days, dtype=dtype, device=device)
        )
        frac = min(max(init_dose_mm / max_dose_mm, 1e-4), 1 - 1e-4)
        self.dose_raw = torch.nn.Parameter(
            torch.full((n_days,), float(torch.logit(torch.tensor(frac))), dtype=dtype, device=device)
        )
        self._t = 0

    def reset(self):
        self._t = 0

    def amounts(self):
        gates = straight_through_gate(torch.sigmoid(self.gate_raw))
        return gates * self.max_dose_cm * torch.sigmoid(self.dose_raw)

    def n_on(self):
        return int((torch.sigmoid(self.gate_raw.detach()) > 0.5).sum())

    def forward(self, engine):
        gate = straight_through_gate(torch.sigmoid(self.gate_raw[self._t]))
        dose = self.max_dose_cm * torch.sigmoid(self.dose_raw[self._t])
        self._t = min(self._t + 1, self.gate_raw.numel() - 1)
        return gate * dose


n_days = len(initial["irrigation"])
schedule = OpenLoopSchedule(n_days)
schedule_opt = torch.optim.Adam(schedule.parameters(), lr=0.25)
schedule_history = []
best_schedule = None
best_schedule_loss = float("inf")
n_schedule_steps = 400

print(
    f"open-loop vector length: {n_days} days  "
    f"(gates start off; {schedule.n_on()} on)"
)
for step in range(n_schedule_steps):
    schedule_opt.zero_grad()
    outcome = evaluate_policy(schedule, provider, weather, agromanagement, wlp_config)
    loss = -outcome["reward"]
    amounts = schedule.amounts()
    record = {
        "step": step,
        "loss": scalarize(loss),
        "reward": scalarize(outcome["reward"]),
        "y_fresh": scalarize(outcome["y_fresh"]),
        "applied_mm": scalarize(outcome["applied_mm"]),
        "n_events": outcome["n_events_hard"],
        "max_day_mm": 10.0 * scalarize(amounts.max()),
    }
    schedule_history.append(record)
    if record["loss"] < best_schedule_loss:
        best_schedule_loss = record["loss"]
        best_schedule = {key: value.detach().clone() for key, value in schedule.state_dict().items()}

    loss.backward()
    schedule_opt.step()
    print(
        f"step {step:03d}  on={schedule.n_on():3d}  fresh={record['y_fresh']:5.1f} t/ha  "
        f"applied={record['applied_mm']:5.0f} mm  events={record['n_events']:3d}  "
        f"max day={record['max_day_mm']:.1f} mm  "
        f"R={record['reward']:7.0f} €/ha"
    )

schedule.load_state_dict(best_schedule)
open_loop = evaluate_policy(schedule, provider, weather, agromanagement, wlp_config)
summarize_economics("open-loop schedule", open_loop)
summarize_economics("optimized policy", optimized)


open-loop vector length: 279 days  (gates start off; 0 on)
step 000  on=  0  fresh=  8.6 t/ha  applied=    0 mm  events=  0  max day=0.0 mm  R=    344 €/ha
step 001  on=  0  fresh=  8.6 t/ha  applied=    0 mm  events=  0  max day=0.0 mm  R=    344 €/ha
step 002  on=  0  fresh=  8.6 t/ha  applied=    0 mm  events=  0  max day=0.0 mm  R=    344 €/ha
step 003  on=  0  fresh=  8.6 t/ha  applied=    0 mm  events=  0  max day=0.0 mm  R=    344 €/ha
step 004  on=  0  fresh=  8.6 t/ha  applied=    0 mm  events=  0  max day=0.0 mm  R=    344 €/ha
step 005  on=  0  fresh=  8.6 t/ha  applied=    0 mm  events=  0  max day=0.0 mm  R=    344 €/ha
step 006  on=  0  fresh=  8.6 t/ha  applied=    0 mm  events=  0  max day=0.0 mm  R=    344 €/ha
step 007  on=  5  fresh=  8.6 t/ha  applied=    0 mm  events=  0  max day=0.0 mm  R=    344 €/ha
step 008  on=  6  fresh= 16.8 t/ha  applied=  125 mm  events=  5  max day=25.0 mm  R=    422 €/ha
step 009  on=  6  fresh= 16.8 t/ha  applied=  146 mm  events=  6  m

In [ ]:
sched_hist = pd.DataFrame(schedule_history)
ol_frame = open_loop["frame"]

fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
axes[0].plot(sched_hist["step"], sched_hist["y_fresh"], marker="o", color="C2")
axes[0].axhline(pp_fresh, color="0.3", linestyle="--", label="potential")
axes[0].axhline(scalarize(optimized["y_fresh"]), color="C0", linestyle="-", label="closed-loop policy")
axes[0].axhline(scalarize(no_stress["y_fresh"]), color="C3", linestyle="-.", label="no-stress")
axes[0].axhline(rainfed_fresh, color="0.3", linestyle=":", label="rainfed")
axes[0].set_xlabel("step")
axes[0].set_ylabel("fresh yield (t ha$^{-1}$)")
axes[0].legend(fontsize=8)

axes[1].plot(sched_hist["step"], sched_hist["reward"], marker="o", color="C2")
axes[1].axhline(scalarize(optimized["reward"]), color="C0", linestyle="-", label="closed-loop policy")
axes[1].axhline(episode_reward(rainfed_df["TWSO"].iloc[-1], zero_amounts), color="0.3", linestyle=":", label="rainfed")
axes[1].set_xlabel("step")
axes[1].set_ylabel("reward (€ ha$^{-1}$)")
axes[1].legend(fontsize=8)

axes[2].plot(sched_hist["step"], sched_hist["n_events"], marker="o", color="C3")
axes[2].set_xlabel("step")
axes[2].set_ylabel("days with $a_t > 15$ mm")

fig.suptitle("Adam on the open-loop daily irrigation vector", y=1.02)
fig.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(9, 3.2))
ax.bar(ol_frame.index, ol_frame["irrigation"], width=1.0, color="C2", alpha=0.85, label="open-loop vector")
ax.plot(
    optimized["frame"].index,
    optimized["frame"]["irrigation"],
    color="C0",
    linewidth=0.9,
    alpha=0.8,
    label="closed-loop policy",
)
ax.set_ylabel("irrigation (mm day$^{-1}$)")
ax.set_xlabel("day")
ax.legend()
ax.set_title("Daily irrigation: open-loop vector versus closed-loop policy")
fig.tight_layout()
plt.show()


The open-loop vector starts with every gate off and has to *flip days on* to 25 mm events. That avoids the 5 mm cost cliff that trapped a continuous drizzle. The search is still harder than the two-parameter policy: early steps often remain rainfed until a few gates cross 0.5, after which yield and irrigation jump together. A long Adam run gives those gates time to appear, then to be pruned if they do not pay. The best $R$ is kept, because a straight-through tape can walk off a good calendar into rainfed.

The no-stress rule is not competing for the same objective. It answers a different question — *what irrigation keeps RFTRA at 1?* — and then we price that schedule after the fact.


In [ ]:
comparison = pd.DataFrame(
    [
        comparison_row("rainfed", rainfed_df["TWSO"].iloc[-1], zero_amounts),
        comparison_row("initial policy", initial["twso"], initial["amounts_cm"]),
        comparison_row("optimized closed-loop", optimized["twso"], optimized["amounts_cm"]),
        comparison_row("no-stress (RFTRA=1)", no_stress["twso"], no_stress["amounts_cm"]),
        comparison_row("optimized open-loop", open_loop["twso"], open_loop["amounts_cm"]),
        comparison_row("potential production", pp_df["TWSO"].iloc[-1], None, potential=True),
    ],
    index=[
        "rainfed",
        "initial policy",
        "optimized closed-loop policy",
        "no-stress (RFTRA=1)",
        "optimized open-loop vector",
        "potential production",
    ],
)
comparison = with_min_rftra(rainfed_df, "rainfed", comparison)
comparison = with_min_rftra(initial["frame"], "initial policy", comparison)
comparison = with_min_rftra(optimized["frame"], "optimized closed-loop policy", comparison)
comparison = with_min_rftra(no_stress["frame"], "no-stress (RFTRA=1)", comparison)
comparison = with_min_rftra(open_loop["frame"], "optimized open-loop vector", comparison)
comparison.round(2)


## 9. Challenges, and what this notebook does not solve

Differentiable decision optimization inherits the difficulties of long-horizon control.

**Weak late gradients.** An irrigation in April affects `TWSO` in autumn only through the intervening water and carbon balances. If the policy starts near "never irrigate", those gradients are easy to lose and the optimizer may stay at the rainfed solution. The structured event policy used here is a partial remedy: a dryness threshold is an agronomic inductive bias, so even a modest initialization already applies water and produces a usable gradient.

**Events versus a daily trickle.** Farmers irrigate on a handful of days with 15–40 mm. A sigmoid gate applies a little water on many days, which the event fee then punishes. The straight-through estimator makes the forward pass look like {0, dose}, which is closer to {0, 10, 20, 30} mm than a continuous pump, but it is not a true categorical action: the backward pass is a biased surrogate. Gumbel-softmax, a weekly decision, or an RL algorithm such as PPO or SAC on a discrete action set would be more literal farmer decisions, at the cost of a harder (less smooth) optimization problem.

**Cost structure.** €1/mm plus €25 per event is an operating-cost split of the BBRO hose-reel figure (~€50 per 25 mm), not a farm invoice and not capital recovery. Water alone is cheaper; labour to move a reel is not. Changing `P`, `c_mm` or `C_event` changes the policy. The point of the euro objective is the *trade-off*: irrigate when extra beet value exceeds cost. The no-stress rule shows the other extreme — keep `RFTRA` at 1 regardless of that trade-off.

**This site is not a 90 t/ha crop.** WOFOST `TWSO` is dry biomass; at 23% dry matter the potential here is about 50 t/ha fresh. Commercial arithmetic such as 90 × 40 − 150 × 1 = €3450/ha has the right *shape* for an RL experiment on a high-yielding crop, but it would overstate profit on this test YAML.

**Open-loop versus closed-loop.** Optimizing $(u_t)$ for a known weather year is the most flexible differentiable control problem in this notebook. It is also the least transferable: the vector memorizes 2010 rainfall. A policy $\pi_\omega(s, x)$ has far fewer parameters and is the object one would actually take to a new season. Adding a weather *forecast* to the state, as one would in RL, is a natural extension; this notebook only uses the current day's rain.

**Capacity versus the STE tape.** A larger controller class *contains* the old MAD rule, so a global maximiser of $R$ cannot do worse than a two-parameter schedule. Adam on a straight-through tape is not that maximiser. Extra weights can leave a good discrete event plan and fall into rainfed, where irrigation gradients vanish and more steps do not climb back. That rainfed point is worse on true $R$; it is a trap of the surrogate gradient, not a better local minimum of the euro objective. The residual is therefore a small correction on a MAD tree, not a blank MLP, and training keeps the best $R$ rather than the last step.

The point of the example is not an operational irrigation scheduler. It is to show that once WOFOST is differentiable, management can be optimized with the same gradient tape as parameters or hybrid modules, with an interpretable €/ha objective and with consequences that remain visible in the crop's physiological states.
